# PAM50 benchmark v1.2: METABRIC + SCAN-B (GSE96058), within-cohort and cross-cohort

Runtime > Run all. Heavy steps save checkpoints in Drive, so if Colab disconnects just Run all again and finished parts are skipped.

* Both cohorts are restricted to genes measured in both.
* The full benchmark (same arms, same 5 seeds, 70/15/15) runs inside METABRIC and inside SCAN-B.
* Every model trained in one cohort also predicts **every** 4-class tumour of the other cohort (transfer).
* genefu PAM50 (no training) is the reference in each target cohort.

Arms: `pam50_centroid_v1` (within-cohort only), `pam50_genefu`, `lr_pam50genes`, `lr_full3000`, `compact_TOPk` (k = 20, 40, 50, 80, 120).

In [ ]:
# Cell 1 — Setup + parameters
import os, json, time, subprocess, shutil, textwrap, gzip, csv, pickle, urllib.request, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", message="Unknown solver options")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
except ImportError:                                   # local test only
    DATA_DIR = os.environ.get("DATA_DIR", "./EXPORT_FOR_COLAB")

SCANB_DIR = os.path.join(DATA_DIR, "GSE96058")
OUT = os.path.join(DATA_DIR, "ML_RESULTS_v1.2")
for sub in ["", "checkpoints", "genefu", "stability", "biology", "env", "per_seed"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
os.makedirs(SCANB_DIR, exist_ok=True)

SEEDS       = [123, 124, 125, 126, 127]
CLASS_ORDER = ["Basal", "Her2", "LumA", "LumB"]
V           = 3000
C_SEL       = 0.6
PANEL_SIZES = [20, 40, 50, 80, 120]
PRIMARY_K   = 80
C_GRID      = [1e-4, 3e-4, 1e-3, 3e-3, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]   # extended down to 1e-4
STABLE_MIN  = 3
N_BOOT      = 2000
ALIAS       = {"CDCA1": "NUF2", "KNTC2": "NDC80", "ORC6L": "ORC6"}       # old PAM50 symbol -> current
COHORTS     = ["METABRIC", "SCANB"]

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict))},
          open(os.path.join(OUT, "env", "run_config.json"), "w"), indent=2)
print("OUT =", OUT)

In [ ]:
# Cell 2 — R + genefu (installs once per Colab session, ~10-15 min) and PAM50 centroids
if shutil.which("Rscript") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "r-base", "r-base-dev"], check=True)
r_install = '''
options(repos = c(CRAN = "https://cloud.r-project.org"))
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("genefu", quietly = TRUE)) BiocManager::install("genefu", update = FALSE, ask = FALSE)
suppressPackageStartupMessages(library(genefu))
suppressWarnings(try(data(pam50.robust, package = "genefu"), silent = TRUE))
if (!exists("pam50.robust")) data(pam50, package = "genefu")
stopifnot(exists("pam50.robust"))
cent <- pam50.robust$centroids
write.table(data.frame(GENE = toupper(rownames(cent)), cent, check.names = FALSE),
            file.path(Sys.getenv("OUT"), "genefu", "pam50_centroids.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
cat("std =", pam50.robust$std, "| method.cor =", pam50.robust$method.cor, "| genefu", as.character(packageVersion("genefu")), "\n")
'''
fp = os.path.join(OUT, "genefu", "install_and_export.R")
open(fp, "w").write(textwrap.dedent(r_install))
p = subprocess.run(["Rscript", fp], env={**os.environ, "OUT": OUT}, capture_output=True, text=True)
open(os.path.join(OUT, "genefu", "install_and_export.log"), "w").write(p.stdout + p.stderr)
print("\n".join((p.stdout + p.stderr).splitlines()[-3:]))
assert p.returncode == 0, "R step failed, see genefu/install_and_export.log"

cent = pd.read_csv(os.path.join(OUT, "genefu", "pam50_centroids.tsv"), sep="\t")
cent["GENE"] = cent["GENE"].str.upper().str.strip()
cent = cent.set_index("GENE")
print("Centroids:", cent.shape)

In [ ]:
# Cell 3 — Load METABRIC (same inputs as v1.0/v1.1)
def log_check(X, name):
    q50, q99 = np.nanpercentile(X, [50, 99])
    if (np.nanmin(X) >= 0) and (np.nanmax(X) >= 500) and ((q99 + 1e-6) / (q50 + 1e-6) >= 20):
        print(name, ": log1p applied"); return np.log1p(X)
    print(name, ": already log-scale"); return X

y_df = pd.read_csv(os.path.join(DATA_DIR, "y_labels.tsv"), sep="\t")
X_df = pd.read_csv(os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz"), sep="\t", compression="gzip")
df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
assert len(df) == len(y_df) and df["SAMPLE_ID"].is_unique
g_mb = [c for c in df.columns if c not in ("SAMPLE_ID", "y")]
raw = {"METABRIC": dict(X=log_check(df[g_mb].to_numpy(np.float32), "METABRIC"),
                        genes=np.array(g_mb, dtype=str),
                        sid=df["SAMPLE_ID"].astype(str).to_numpy(),
                        y=df["y"].astype(str).to_numpy())}
del X_df, df
print("METABRIC:", raw["METABRIC"]["X"].shape, pd.Series(raw["METABRIC"]["y"]).value_counts().to_dict())

In [ ]:
# Cell 4 — Download (once, cached in Drive) and load SCAN-B GSE96058
GEO = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE96nnn/GSE96058"
FILES = {"expr": f"{GEO}/suppl/GSE96058_gene_expression_3273_samples_and_136_replicates_transformed.csv.gz",
         "sm1":  f"{GEO}/matrix/GSE96058-GPL11154_series_matrix.txt.gz",
         "sm2":  f"{GEO}/matrix/GSE96058-GPL18573_series_matrix.txt.gz"}
for key, url in FILES.items():
    fp = os.path.join(SCANB_DIR, os.path.basename(url))
    if not os.path.exists(fp):
        print("downloading", os.path.basename(url)); urllib.request.urlretrieve(url, fp + ".part"); os.rename(fp + ".part", fp)

def pam50_from_series_matrix(fp):
    titles, labels = None, None
    with gzip.open(fp, "rt", errors="replace") as f:
        for row in csv.reader(f, delimiter="\t"):
            if not row: continue
            if row[0] == "!Sample_title":
                titles = [v.strip() for v in row[1:]]
            elif row[0] == "!Sample_characteristics_ch1" and any("pam50" in v.lower() for v in row[1:]):
                labels = [v.split(":", 1)[1].strip() if ":" in v else "" for v in row[1:]]
    assert titles and labels, f"PAM50 characteristic not found in {fp}"
    return pd.Series(labels, index=titles)

cache = os.path.join(SCANB_DIR, "scanb_cache.npz")
if not os.path.exists(cache):
    lab = pd.concat([pam50_from_series_matrix(os.path.join(SCANB_DIR, os.path.basename(FILES[k]))) for k in ("sm1", "sm2")])
    lab = lab[~lab.index.duplicated()]
    E = pd.read_csv(os.path.join(SCANB_DIR, os.path.basename(FILES["expr"])), index_col=0)
    E = E.loc[:, [c for c in E.columns if not c.endswith("repl")]]                 # drop technical replicates
    E.index = E.index.astype(str).str.upper().str.strip()
    E = E.groupby(level=0).mean()                                                  # collapse duplicate symbols
    keep = [c for c in E.columns if c in lab.index]
    assert len(keep) > 0.9 * E.shape[1], f"only {len(keep)} of {E.shape[1]} expression columns matched series-matrix titles"
    np.savez(cache, X=E[keep].T.to_numpy(np.float32), genes=E.index.to_numpy(str),
             sid=np.array(keep, dtype=str), y=lab.loc[keep].to_numpy(str))
    del E
z = np.load(cache, allow_pickle=False)
raw["SCANB"] = dict(X=log_check(z["X"], "SCANB"), genes=z["genes"].astype(str), sid=z["sid"].astype(str), y=z["y"].astype(str))
print("SCAN-B:", raw["SCANB"]["X"].shape, pd.Series(raw["SCANB"]["y"]).value_counts().to_dict())

In [ ]:
# Cell 5 — Harmonise gene space (current symbols, genes present in both cohorts) + PAM50 indices
def canon(g):
    g = np.char.upper(np.char.strip(np.asarray(g, dtype=str)))
    return np.array([ALIAS.get(x, x) for x in g])

for c in COHORTS:
    raw[c]["genes"] = canon(raw[c]["genes"])
common = sorted(set(raw["METABRIC"]["genes"]) & set(raw["SCANB"]["genes"]))
print("genes in both cohorts:", len(common))

pam_canon = canon(cent.index)                         # centroid genes in current symbols
missing = sorted(set(pam_canon) - set(common))
assert not missing, f"PAM50 genes missing from shared gene space: {missing}"

data = {}
for c in COHORTS:
    r = raw[c]
    pos = {g: i for i, g in enumerate(r["genes"])}
    cols = np.array([pos[g] for g in common])
    m4 = np.isin(r["y"], CLASS_ORDER)
    data[c] = dict(X=r["X"][:, cols], genes=np.array(common), sid=r["sid"], ylab=r["y"],
                   X4=r["X"][m4][:, cols], sid4=r["sid"][m4],
                   y4=np.array([CLASS_ORDER.index(v) for v in r["y"][m4]]))
    print(c, "4-class:", data[c]["X4"].shape, dict(zip(CLASS_ORDER, np.bincount(data[c]["y4"]))))
del raw

gpos = {g: i for i, g in enumerate(common)}
pam_idx = np.array([gpos[g] for g in pam_canon])      # same order as cent.index
C_pam = cent[CLASS_ORDER].to_numpy(np.float32)

In [ ]:
# Cell 6 — Helpers
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, average_precision_score, recall_score
from scipy.stats import binomtest

def split_70_15_15(y, seed):                          # identical to v1.0/v1.1
    idx = np.arange(len(y))
    tr, tmp = next(StratifiedShuffleSplit(1, test_size=0.30, random_state=seed).split(idx, y))
    va_r, te_r = next(StratifiedShuffleSplit(1, test_size=0.50, random_state=seed).split(tmp, y[tmp]))
    return tr, tmp[va_r], tmp[te_r]

class Prep:                                           # median impute + z-score, fitted on the data given to fit()
    def fit(self, X):
        self.imp = SimpleImputer(strategy="median").fit(X)
        self.sc = StandardScaler().fit(self.imp.transform(X)); return self
    def tf(self, X):
        return self.sc.transform(self.imp.transform(X)).astype(np.float32)

def cohort_z(X):                                      # external cohort: label-free z-score within that cohort
    return Prep().fit(X).tf(X)

def lr(C, seed):
    return LogisticRegression(penalty="l2", solver="lbfgs", C=C, class_weight="balanced", max_iter=5000, random_state=seed)

def tune_refit(Xtr, ytr, Xva, yva, seed):            # C on VAL (macro-F1), refit on TRAIN+VAL
    f = {C: f1_score(yva, lr(C, seed).fit(Xtr, ytr).predict(Xva), average="macro") for C in C_GRID}
    C = max(f, key=f.get)
    return lr(C, seed).fit(np.vstack([Xtr, Xva]), np.concatenate([ytr, yva])), C

def softmax(S):
    E = np.exp(S - S.max(1, keepdims=True)); return E / E.sum(1, keepdims=True)

def pam50_centroid_v1(Xtr_raw, Xte_raw):              # v1.0 rule (kept as historical reference)
    A, B = Xtr_raw[:, pam_idx], Xte_raw[:, pam_idx]
    med = np.nanmedian(A, 0); mad = np.nanmedian(np.abs(A - med), 0) + 1e-6
    Z = np.nan_to_num((B - med) / mad)
    Zc = Z - Z.mean(1, keepdims=True); Cc = C_pam - C_pam.mean(0, keepdims=True)
    return softmax((Zc / (np.linalg.norm(Zc, axis=1, keepdims=True) + 1e-8)) @ (Cc / (np.linalg.norm(Cc, axis=0, keepdims=True) + 1e-8)))

def macro_f1_fast(y, p, K=4):
    cm = np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); denom = cm.sum(0) + cm.sum(1)
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0)))

def metrics(y, P):
    yhat = P.argmax(1); yb = label_binarize(y, classes=range(4))
    rec = recall_score(y, yhat, labels=range(4), average=None, zero_division=0)
    f1c = f1_score(y, yhat, labels=range(4), average=None, zero_division=0)
    out = {"macro_f1": f1_score(y, yhat, average="macro"), "bacc": balanced_accuracy_score(y, yhat),
           "macro_roc_auc": roc_auc_score(yb, P, average="macro"), "macro_pr_auc": average_precision_score(yb, P, average="macro")}
    out.update({f"recall_{c}": v for c, v in zip(CLASS_ORDER, rec)})
    out.update({f"f1_{c}": v for c, v in zip(CLASS_ORDER, f1c)})
    return out

def pred_rows(seed, arm, scope, sids, y, P):
    d = pd.DataFrame({"seed": seed, "arm": arm, "scope": scope, "SAMPLE_ID": sids, "y_true": y, "y_pred": P.argmax(1)})
    for q, c in enumerate(CLASS_ORDER): d[f"p_{c}"] = P[:, q]
    return d

In [ ]:
# Cell 7 — Benchmark inside each cohort; every refit model also predicts the whole other cohort
def run_cohort(src, dst):
    ck = os.path.join(OUT, "checkpoints", f"{src}.pkl")
    if os.path.exists(ck):
        print(src, "loaded from checkpoint"); return pickle.load(open(ck, "rb"))
    S, D = data[src], data[dst]
    Xd_z_full = cohort_z(D["X4"])                     # external cohort, label-free z-score (all shared genes)
    rows, preds, panels, tuned, splits = [], [], [], [], []
    t0 = time.time()
    for s in SEEDS:
        tr, va, te = split_70_15_15(S["y4"], s)
        for part, ix in (("train", tr), ("val", va), ("test", te)):
            splits.append(pd.DataFrame({"cohort": src, "seed": s, "part": part, "SAMPLE_ID": S["sid4"][ix],
                                        "y": np.array(CLASS_ORDER)[S["y4"][ix]]}))
        Xtr_r, Xva_r, Xte_r = S["X4"][tr], S["X4"][va], S["X4"][te]
        ytr, yva, yte = S["y4"][tr], S["y4"][va], S["y4"][te]
        within, ext, Cs = {}, {}, {}

        within["pam50_centroid_v1"] = pam50_centroid_v1(Xtr_r, Xte_r)

        def fit_arm(name, cols):
            pr = Prep().fit(Xtr_r[:, cols])
            m, Cs[name] = tune_refit(pr.tf(Xtr_r[:, cols]), ytr, pr.tf(Xva_r[:, cols]), yva, s)
            # TEST uses the TRAIN-fitted preprocessing (as v1.1); external uses its own cohort z-score
            within[name] = m.predict_proba(pr.tf(Xte_r[:, cols]))
            ext[name] = m.predict_proba(Xd_z_full[:, cols])

        fit_arm("lr_pam50genes", pam_idx)
        v_idx = np.argsort(np.var(SimpleImputer(strategy="median").fit_transform(Xtr_r), 0))[::-1][:V]
        fit_arm("lr_full3000", v_idx)
        pr = Prep().fit(Xtr_r[:, v_idx])
        sel = LogisticRegression(penalty="l1", solver="saga", C=C_SEL, class_weight="balanced",
                                 max_iter=1500, tol=1e-3, random_state=s).fit(pr.tf(Xtr_r[:, v_idx]), ytr)
        rank = v_idx[np.argsort(np.abs(sel.coef_).mean(0))[::-1]]
        for k in PANEL_SIZES:
            fit_arm(f"compact_TOP{k}", rank[:k])
        panels.append(pd.DataFrame({"cohort": src, "seed": s, "rank": np.arange(1, PRIMARY_K + 1),
                                    "GENE": S["genes"][rank[:PRIMARY_K]]}))
        tuned.append({"cohort": src, "seed": s, **{f"C_{a}": c for a, c in Cs.items()},
                      "C_at_grid_edge": ";".join(a for a, c in Cs.items() if c in (C_GRID[0], C_GRID[-1]))})

        for arm, P in within.items():
            rows.append({"train_cohort": src, "test_cohort": src, "scope": "within", "seed": s, "arm": arm, **metrics(yte, P)})
            preds.append(pred_rows(s, arm, f"{src}_within", S["sid4"][te], yte, P))
        for arm, P in ext.items():
            rows.append({"train_cohort": src, "test_cohort": dst, "scope": "transfer", "seed": s, "arm": arm, **metrics(D["y4"], P)})
            preds.append(pred_rows(s, arm, f"{src}_to_{dst}", D["sid4"], D["y4"], P))
        print(f"{src} seed {s} ({(time.time() - t0) / 60:.1f} min) | TOP80 within={f1_score(yte, within[f'compact_TOP{PRIMARY_K}'].argmax(1), average='macro'):.3f}"
              f" | PAM50-gene LR within={f1_score(yte, within['lr_pam50genes'].argmax(1), average='macro'):.3f}")
    res = dict(rows=pd.DataFrame(rows), preds=pd.concat(preds), panels=pd.concat(panels),
               tuned=pd.DataFrame(tuned), splits=pd.concat(splits))
    pickle.dump(res, open(ck, "wb"))
    return res

R = {"METABRIC": run_cohort("METABRIC", "SCANB"), "SCANB": run_cohort("SCANB", "METABRIC")}
tuned = pd.concat([R[c]["tuned"] for c in COHORTS]); splits = pd.concat([R[c]["splits"] for c in COHORTS])
tuned.to_csv(os.path.join(OUT, "per_seed", "tuned_hyperparameters.tsv"), sep="\t", index=False)
splits.to_csv(os.path.join(OUT, "splits_by_sample_id.tsv"), sep="\t", index=False)
print(tuned[["cohort", "seed", "C_at_grid_edge"]])

In [ ]:
# Cell 8 — genefu PAM50 (intrinsic.cluster.predict, pam50.robust) in R
#   per seed on each TEST set alone, plus once on each full labelled cohort (= transfer reference, no training)
gdir = os.path.join(OUT, "genefu")
calls_fp = os.path.join(gdir, "genefu_calls.tsv")
if not os.path.exists(calls_fp):
    for c in COHORTS:
        Xg = pd.DataFrame(data[c]["X"][:, pam_idx], columns=cent.index)     # genefu's own symbols
        Xg.insert(0, "SAMPLE_ID", data[c]["sid"])
        Xg.to_csv(os.path.join(gdir, f"pam50_expr_{c}.tsv"), sep="\t", index=False)
    splits.query("part == 'test'")[["cohort", "seed", "SAMPLE_ID"]].to_csv(os.path.join(gdir, "test_sets.tsv"), sep="\t", index=False)
    r_code = r'''
    suppressPackageStartupMessages(library(genefu))
    suppressWarnings(try(data(pam50.robust, package = "genefu"), silent = TRUE))
    if (!exists("pam50.robust")) data(pam50, package = "genefu")
    d <- Sys.getenv("GDIR")
    tests <- read.delim(file.path(d, "test_sets.tsv"))
    run <- function(M, tag, cohort) {
      annot <- data.frame(probe = colnames(M), EntrezGene.ID = NA, row.names = colnames(M))
      r <- intrinsic.cluster.predict(sbt.model = pam50.robust, data = M, annot = annot,
                                     do.mapping = FALSE, do.prediction.strength = FALSE, verbose = FALSE)
      data.frame(cohort = cohort, run = tag, SAMPLE_ID = rownames(M), call = as.character(r$subtype),
                 r$subtype.proba, check.names = FALSE)
    }
    res <- NULL
    for (co in c("METABRIC", "SCANB")) {
      X <- read.delim(file.path(d, paste0("pam50_expr_", co, ".tsv")), check.names = FALSE)
      rownames(X) <- X$SAMPLE_ID; X$SAMPLE_ID <- NULL; X <- as.matrix(X)
      stopifnot(all(rownames(pam50.robust$centroids) %in% colnames(X)))
      res <- rbind(res, run(X, "full_cohort", co))
      for (s in unique(tests$seed[tests$cohort == co])) {
        ids <- as.character(tests$SAMPLE_ID[tests$cohort == co & tests$seed == s])
        res <- rbind(res, run(X[ids, , drop = FALSE], paste0("seed_", s), co))
      }
    }
    write.table(res, file.path(d, "genefu_calls.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
    cat("genefu calls written:", nrow(res), "\n")
    '''
    fp = os.path.join(gdir, "run_genefu.R")
    open(fp, "w").write(textwrap.dedent(r_code))
    p = subprocess.run(["Rscript", fp], env={**os.environ, "GDIR": gdir}, capture_output=True, text=True)
    open(os.path.join(gdir, "run_genefu.log"), "w").write(p.stdout + p.stderr)
    print((p.stdout + p.stderr)[-800:])
    assert p.returncode == 0, "genefu step failed, see genefu/run_genefu.log"

gf = pd.read_csv(calls_fp, sep="\t")
GF_UNDEFINED = []                                     # samples where genefu returned no valid probability
def gf_probs(sub, tag=""):
    P = sub[CLASS_ORDER].to_numpy(float)
    bad = ~np.isfinite(P).all(1) | (np.nan_to_num(P).sum(1) <= 0)
    if bad.any():                                     # keep genefu's own crisp call (max correlation) for these
        calls = sub["call"].astype(str).to_numpy()[bad]
        P[bad] = [np.eye(4)[CLASS_ORDER.index(k)] if k in CLASS_ORDER else np.full(4, 0.25) for k in calls]
        GF_UNDEFINED.extend({"run": tag, "SAMPLE_ID": i, "genefu_call": k} for i, k in zip(sub.index[bad], calls))
    return P / P.sum(1, keepdims=True)

g_rows, g_preds, conc = [], [], []
for c in COHORTS:
    other = [o for o in COHORTS if o != c][0]
    for s in SEEDS:
        te = splits.query("cohort == @c and seed == @s and part == 'test'")
        sub = gf[(gf.cohort == c) & (gf.run == f"seed_{s}")].set_index("SAMPLE_ID").loc[te.SAMPLE_ID]
        y = np.array([CLASS_ORDER.index(v) for v in te.y]); P = gf_probs(sub, f"{c}_seed_{s}")
        g_rows.append({"train_cohort": "none", "test_cohort": c, "scope": "within", "seed": s, "arm": "pam50_genefu",
                       **metrics(y, P), "n_called_Normal": int((sub["call"] == "Normal").sum())})
        g_preds.append(pred_rows(s, "pam50_genefu", f"{c}_within", te.SAMPLE_ID.to_numpy(), y, P))
    full = gf[(gf.cohort == c) & (gf.run == "full_cohort")].set_index("SAMPLE_ID")
    sub = full.loc[data[c]["sid4"]]; P = gf_probs(sub, f"{c}_full_cohort")
    g_rows.append({"train_cohort": "none", "test_cohort": c, "scope": "transfer", "seed": 0, "arm": "pam50_genefu",
                   **metrics(data[c]["y4"], P), "n_called_Normal": int((sub["call"] == "Normal").sum())})
    g_preds.append(pred_rows(0, "pam50_genefu", f"{other}_to_{c}", data[c]["sid4"], data[c]["y4"], P))
    ct = pd.crosstab(pd.Series(data[c]["ylab"], index=data[c]["sid"], name="published_PAM50"),
                     full.loc[data[c]["sid"], "call"].rename("genefu_call"))
    ct.to_csv(os.path.join(gdir, f"genefu_vs_published_crosstab_{c}.tsv"), sep="\t")
    conc.append({"cohort": c, "n_4class": len(sub), "exact_agreement_5call": float((sub["call"].to_numpy() == np.array(CLASS_ORDER)[data[c]["y4"]]).mean())})
    print(c); print(ct)
pd.DataFrame(conc).to_csv(os.path.join(gdir, "genefu_full_cohort_agreement.tsv"), sep="\t", index=False)
pd.DataFrame(GF_UNDEFINED, columns=["run", "SAMPLE_ID", "genefu_call"]).to_csv(
    os.path.join(gdir, "genefu_undefined_probabilities.tsv"), sep="\t", index=False)
print("tumours with undefined genefu probabilities (crisp call used):",
      pd.DataFrame(GF_UNDEFINED, columns=["run", "SAMPLE_ID", "genefu_call"]).groupby("run").size().to_dict())

In [ ]:
# Cell 9 — All metrics, summaries, paired tests, luminal confusions
per_seed = pd.concat([R[c]["rows"] for c in COHORTS] + [pd.DataFrame(g_rows)], ignore_index=True)
preds = pd.concat([R[c]["preds"] for c in COHORTS] + g_preds, ignore_index=True)
per_seed.to_csv(os.path.join(OUT, "per_seed", "per_seed_metrics_all.tsv"), sep="\t", index=False)
preds.to_csv(os.path.join(OUT, "per_seed", "predictions_all.tsv.gz"), sep="\t", index=False)

ARMS = ["pam50_centroid_v1", "pam50_genefu", "lr_pam50genes", "lr_full3000"] + [f"compact_TOP{k}" for k in PANEL_SIZES]
mcols = ["macro_f1", "bacc", "macro_roc_auc", "macro_pr_auc"] + [f"{m}_{c}" for m in ("recall", "f1") for c in CLASS_ORDER]
summ = per_seed.groupby(["scope", "test_cohort", "arm"])[mcols].agg(["mean", "std"])
summ.columns = [f"{a}_{b}" for a, b in summ.columns]
summ = summ.reset_index()
summ["arm"] = pd.Categorical(summ["arm"], ARMS, ordered=True)
summ = summ.sort_values(["scope", "test_cohort", "arm"])
summ.to_csv(os.path.join(OUT, "summary_mean_sd.tsv"), sep="\t", index=False)
print(summ.pivot_table(index="arm", columns=["scope", "test_cohort"], values="macro_f1_mean", observed=True).round(3))

# paired tests on identical samples: reference arm against every other arm
rng = np.random.default_rng(2026)
def paired(a, b):
    a = a.set_index("SAMPLE_ID"); b = b.set_index("SAMPLE_ID").loc[a.index]
    y, pa, pb = a.y_true.to_numpy(), a.y_pred.to_numpy(), b.y_pred.to_numpy()
    n = len(y); idx = rng.integers(0, n, (N_BOOT, n))
    bs = np.array([macro_f1_fast(y[i], pa[i]) - macro_f1_fast(y[i], pb[i]) for i in idx])
    n01, n10 = int(((pa == y) & (pb != y)).sum()), int(((pa != y) & (pb == y)).sum())
    return {"delta_macro_f1": macro_f1_fast(y, pa) - macro_f1_fast(y, pb),
            "ci_low": np.percentile(bs, 2.5), "ci_high": np.percentile(bs, 97.5),
            "ref_right_comp_wrong": n01, "ref_wrong_comp_right": n10,
            "mcnemar_p": binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0}

tests = []
for scope in preds.scope.unique():
    for ref in ["lr_pam50genes", f"compact_TOP{PRIMARY_K}"]:
        for s in SEEDS:
            A = preds[(preds.scope == scope) & (preds.arm == ref) & (preds.seed == s)]
            if A.empty: continue
            for comp in ARMS:
                if comp == ref: continue
                B = preds[(preds.scope == scope) & (preds.arm == comp) & (preds.seed.isin([s, 0]))]
                if B.empty: continue
                B = B[B.seed == (s if s in set(B.seed) else 0)]
                tests.append({"scope": scope, "reference": ref, "comparator": comp, "seed": s, **paired(A, B)})
tests = pd.DataFrame(tests)
tests.to_csv(os.path.join(OUT, "paired_tests.tsv"), sep="\t", index=False)
print(tests.groupby(["scope", "reference", "comparator"]).agg(
    mean_delta=("delta_macro_f1", "mean"), seeds_ci_above0=("ci_low", lambda x: int((x > 0).sum())),
    seeds_ci_below0=("ci_high", lambda x: int((x < 0).sum()))).round(3).to_string())

lum = []
for (scope, arm), q in preds.groupby(["scope", "arm"]):
    cm = np.bincount(q.y_true * 4 + q.y_pred, minlength=16).reshape(4, 4)
    lum.append({"scope": scope, "arm": arm, "LumA_to_LumB": cm[2, 3], "LumB_to_LumA": cm[3, 2],
                "LumB_to_Her2": cm[3, 1], "LumA_to_Basal": cm[2, 0], "n_LumA": cm[2].sum(), "n_LumB": cm[3].sum()})
pd.DataFrame(lum).to_csv(os.path.join(OUT, "luminal_confusions.tsv"), sep="\t", index=False)

In [ ]:
# Cell 10 — Panel stability within each cohort + cross-cohort reproducibility
pam_set = set(pam_canon)
stable, summ_rows = {}, []
for c in COHORTS:
    pan = R[c]["panels"]
    freq = pan.groupby("GENE").agg(seed_count=("seed", "nunique"), mean_rank=("rank", "mean")).reset_index() \
              .sort_values(["seed_count", "mean_rank"], ascending=[False, True])
    freq["in_PAM50"] = freq.GENE.isin(pam_set)
    freq.to_csv(os.path.join(OUT, "stability", f"gene_inclusion_frequency_{c}.tsv"), sep="\t", index=False)
    stable[c] = freq[freq.seed_count >= STABLE_MIN]
    stable[c].to_csv(os.path.join(OUT, "stability", f"stable_panel_ge{STABLE_MIN}of{len(SEEDS)}_{c}.tsv"), sep="\t", index=False)
    sets = [set(g) for _, g in pan.groupby("seed").GENE]
    jac = [len(x & y) / len(x | y) for i, x in enumerate(sets) for y in sets[i + 1:]]
    summ_rows.append({"cohort": c, "unique_genes": len(freq),
                      **{f"in_{k}_of_{len(SEEDS)}": int((freq.seed_count == k).sum()) for k in range(1, len(SEEDS) + 1)},
                      "stable_size": len(stable[c]), "stable_in_PAM50": int(stable[c].in_PAM50.sum()),
                      "mean_pairwise_jaccard_TOP80": float(np.mean(jac)),
                      "always_selected": ";".join(sorted(freq[freq.seed_count == len(SEEDS)].GENE))})
a, b = set(stable["METABRIC"].GENE), set(stable["SCANB"].GENE)
summ_rows.append({"cohort": "METABRIC_and_SCANB_stable_overlap", "unique_genes": len(a | b), "stable_size": len(a & b),
                  "stable_in_PAM50": len((a & b) & pam_set), "mean_pairwise_jaccard_TOP80": len(a & b) / max(1, len(a | b)),
                  "always_selected": ";".join(sorted(a & b))})
stab = pd.DataFrame(summ_rows)
stab.to_csv(os.path.join(OUT, "stability", "stability_summary.tsv"), sep="\t", index=False)
print(stab.T.to_string())

In [ ]:
# Cell 11 — METABRIC stable panel: coefficients (seed-123 split, refit TRAIN+VAL) and enrichment
import requests
from scipy.stats import hypergeom, false_discovery_control

c, s = "METABRIC", 123
sp = splits.query("cohort == @c and seed == @s and part != 'test'")
pos = {x: i for i, x in enumerate(data[c]["sid4"])}
tv = np.array([pos[x] for x in sp.SAMPLE_ID])
gi = np.array([gpos[g] for g in stable[c].GENE])
pr = Prep().fit(data[c]["X4"][tv][:, gi])
m = lr(1.0, s).fit(pr.tf(data[c]["X4"][tv][:, gi]), data[c]["y4"][tv])
coef = pd.DataFrame(m.coef_.T, index=stable[c].GENE.to_numpy(), columns=CLASS_ORDER)
coef = coef.loc[coef.abs().max(1).sort_values(ascending=False).index]
coef.to_csv(os.path.join(OUT, "biology", "stable_panel_LR_coefficients_METABRIC_seed123.tsv"), sep="\t")

def get_library(lib):
    fp = os.path.join(OUT, "biology", f"{lib}.gmt")
    if not os.path.exists(fp):
        r = requests.get("https://maayanlab.cloud/Enrichr/geneSetLibrary", params={"mode": "text", "libraryName": lib}, timeout=300)
        r.raise_for_status(); open(fp, "w").write(r.text)
    sets = {}
    for line in open(fp):
        p = line.rstrip("\n").split("\t")
        if len(p) > 2: sets[p[0]] = {g.split(",")[0].strip().upper() for g in p[2:] if g.strip()}
    return sets

bg = set()                                            # genes the selector could choose: train-only top-V per seed
for s in SEEDS:
    tr = np.array([pos[x] for x in splits.query("cohort == @c and seed == @s and part == 'train'").SAMPLE_ID])
    bg |= set(data[c]["genes"][np.argsort(np.var(SimpleImputer(strategy="median").fit_transform(data[c]["X4"][tr]), 0))[::-1][:V]])
query = set(stable[c].loc[~stable[c].in_PAM50, "GENE"]) & bg
res = []
for lib in ["Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2023"]:
    gsets = {t: g & bg for t, g in get_library(lib).items()}
    gsets = {t: g for t, g in gsets.items() if 5 <= len(g) <= 500}
    rows = [{"library": lib, "Term": t, "Overlap": f"{len(query & g)}/{len(g)}", "Genes": ";".join(sorted(query & g)),
             "P-value": hypergeom.sf(len(query & g) - 1, len(bg), len(g), len(query))} for t, g in gsets.items() if query & g]
    t = pd.DataFrame(rows)
    t["Adjusted P-value"] = false_discovery_control(np.r_[t["P-value"], np.ones(len(gsets) - len(t))], method="bh")[:len(t)]
    res.append(t.assign(terms_tested=len(gsets)))
enr = pd.concat(res).sort_values("Adjusted P-value")
enr.to_csv(os.path.join(OUT, "biology", "enrichment_METABRIC_stable_nonPAM50.tsv"), sep="\t", index=False)
print(f"query {len(query)} | background {len(bg)}")
print(enr[["library", "Term", "Overlap", "Adjusted P-value"]].head(8).to_string(index=False))

In [ ]:
# Cell 12 — Environment record + output list
import sklearn, scipy, platform
open(os.path.join(OUT, "env", "python_versions.txt"), "w").write(
    f"python {platform.python_version()}\nnumpy {np.__version__}\npandas {pd.__version__}\n"
    f"scikit-learn {sklearn.__version__}\nscipy {scipy.__version__}\n")
for root, _, files in sorted(os.walk(OUT)):
    for f in sorted(files):
        if not f.endswith(".gmt") and "checkpoints" not in root:
            print(os.path.relpath(os.path.join(root, f), OUT))